# Biohub - Cell Tracking During Development / `Biohub M001 ens3 sm6 sim2`

- **コンペ**: [Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)
- **原著notebook**: [Biohub M001 ens3 sm6 sim2](https://www.kaggle.com/code/xiaoleilian/biohub-m001-ens3-sm6-sim2)
- **原著者**: XIAOLEI LIAN (xiaoleilian) ・ 61 votes（Silver）・ Apache 2.0 ・ Version 1 / ランタイム 17分2秒（GPU T4 x2）
- **スコア**: Public 0.917 / Best 0.917（作者自身のVAL-24内部指標では 0.8623）

> ⚠️ これは**学習目的の解説付き写し**です。コード本体は原著のまま変更していませんが、実行結果（outputs）は含みません。
> コードの著作権は原著者に帰属します（Apache 2.0）。

## 手法の概要

発生中の胚を撮影した**3Dタイムラプス動画**から、(1) 各フレームで細胞の中心を検出し、(2) それをフレーム間でつないで
**細胞系譜（リネージ）グラフ**を作る、という2段構えのパイプラインです。
検出は「3D U-Netが出力する細胞中心のヒートマップ」、リンクは「ハンガリー法（線形割当）」という古典的な組み合わせですが、
このnotebookの価値は**その両方に少しずつ足された改良の積み上げ方**にあります。作者は内部検証スコアの推移を明記しています:

```
tta4f 0.8547 → ens3 0.8584 → +sm6 0.8601 → +sim2 0.8623
```

つまり「1つの大発明」ではなく、**+0.004 / +0.002 / +0.002 という小さな改善を、毎回検証して積む**という進め方です。
初心者にとってはここが一番の学びどころで、モデルを作り直すより先に、**推論の平均の取り方・後処理の閾値**で伸びる余地がまだあることを示しています。

タイトルの略語はそのまま改良の履歴になっています。

| 略語 | 意味 |
|---|---|
| `ens3` | 3モデルのアンサンブル（bright / tophat / v2-tophat-b32） |
| `sm6` | short-track filter の閾値を4→6に変更（短すぎるトラックを捨てる） |
| `sim2` | リンクのコストに**見た目の類似度（similarity）**を足した（係数2.0） |
| `tta4f` | 4通りの反転（flip）によるTest-Time Augmentation |

## 評価指標

- **タスク**: 3D動画中の細胞を毎フレーム検出し、時間方向にリンクして、**細胞分裂（1つの親→2つの娘）を含む系譜グラフ**を復元する。
  提出物は「ノード行（各時刻の細胞座標）」と「エッジ行（親→子のリンク）」を混在させたCSV。
- **指標**: 公式の `tracking_cellmot` 系スコア。中身は
  **ノードの検出精度**（正しい位置に細胞を見つけられたか）と
  **エッジ（リンク）のJaccard**（つなぎ方が正しいか）、さらに**分裂イベントの正誤**を合わせた複合指標です。
  Jaccard = 共通部分 ÷ 和集合 なので、「つなぎ忘れ（FN）」も「余計につないだ（FP）」も両方減点されます。
- **なぜこの指標か**: 細胞追跡では「1フレームだけ当てる」ことに意味がなく、**時間方向に一貫した1本の軌跡**と、
  **分裂という構造の変化**を正しく捉えられるかが生物学的な価値そのものだからです。
  検出だけを測る指標（例えばmAP）だと、フレームごとに別の細胞に飛び移る滅茶苦茶な軌跡でも高得点になってしまいます。
- **このnotebookの設計と指標の対応**:
  - **検出のFNを減らす** → 3モデル＋4方向フリップTTA（計12パス）でヒートマップを平均。ノイズによる取りこぼしを減らす。
  - **エッジのFPを減らす** → ハンガリー法を「厳しいゲート(6µm)で1回 → 余りだけ緩いゲート(10µm)でもう1回」の2段階にし、
    さらに `sim2` で**明るさ（検出スコア）が急に変わる組み合わせにペナルティ**をかけて、別細胞への乗り移りを抑える。
  - **短いゴミ軌跡を消す** → `SHORT_MIN=6`。ただし分裂に関わるノードは保護する。
  - **分裂の取りこぼし（FN）を埋める** → 基本のハンガリー法は必ず1対1なので、**分裂で生まれた2人目の娘が原理的に絶対リンクされない**。
    そこを「PATCH 1: safe divisions」で後から救済している（作者の計測で分裂の TP/FP/FN が 0/0/14 → 6/31/8 に改善）。


## 原著者による説明（原文ママ）

# 🧬 Biohub Cell Tracking — 3D U-Net (2 models)

Inference/submission kernel: loads 2 trained 3D U-Nets, predicts a per-voxel cell-centre heatmap for every
frame (heatmaps AVERAGED — each model with its OWN preprocessing), turns peaks into detections, links them across time (two-pass µm-gated Hungarian),
writes `submission.csv`.
Weights = **`unet3d_bright.pt` (preproc: none), `unet3d_traintophat.pt` (preproc: tophat)** from the attached
`biohub-unet3d-weights` dataset; peak threshold **0.15**; own repair **True**.
Fully offline; conv3d GPU probe → CPU fallback (prefer T4 x2).
---
**v6 — 3-model ensemble (frozen bright+tophat + v2-tophat b32) + flip-quartet TTA
+ SHORT_MIN=6 + link appearance cost**. Detection: per-branch mean of 4 flip views
(id/flipY/flipX/flipY.flipX), LOGITS averaged before sigmoid; three branch heatmaps
averaged (equal weights). Linker: two-pass Hungarian with an extra appearance cost
(cost += 2.0 * |logit(score_prev) - logit(score_curr)| um, VAL-24 robust +0.0016-0.0022);
post-link patches (safe divisions + 1f gap snap-only) unchanged; short-track filter 4 -> 6.
VAL-24 (official `tracking_cellmot` metric): **0.8623** (tta4f 0.8547 -> ens3 0.8584 -> +sm6 0.8601 -> +sim2).


## セル1: 環境設定と「壊れたGPUを検出してCPUに逃げる」防御

**何をしているか**: ライブラリを読み込み、GPUが本当に使えるかを**実際に小さな3D畳み込みを1回走らせて**確かめてから
`DEVICE` を決めています。そのあと、パイプライン全体を制御する定数を一気に定義します。

**なぜそうするのか**:
- `torch.cuda.is_available()` は「GPUが刺さっているか」しか見ません。Kaggleでは古いGPU（P100 = sm_60）が割り当てられることがあり、
  そこでは3D畳み込み（`Conv3d`）が実行時に落ちます。**本番の推論が終盤で落ちると提出できずに終わる**ので、
  最初に捨てても惜しくない小さなテンソルで試し、ダメならCPUに切り替えます（GPU実行プローブ）。
- `SCALE = [1.625, 0.40625, 0.40625]` は**1ボクセルの実寸（マイクロメートル）**です。Z方向だけ4倍粗い**異方性**データなので、
  「距離」を計算するときは必ずこの係数を掛けて**物理単位（µm）に直してから**比べます。
  ピクセル数のまま距離を測ると、Z方向の1歩がXY方向の4歩ぶんになってしまい、リンクの判定が壊れます。
- `CAND_THR=0.05`（候補として拾う下限）と、後で使う `0.15`（本採用の閾値）の**2段構え**になっている点にも注目。
  低い閾値で拾った弱い候補は「ギャップ補完（見失った1フレームを埋める）」の裏付けとしてだけ使われます。

**用語**: *ボクセル* = 3D版のピクセル。*異方性(anisotropic)* = 軸によって解像度が違うこと。


In [ ]:

import os, json, glob, time, gc
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage.feature import peak_local_max

DEVICE = "cpu"
if torch.cuda.is_available():
    try:
        _p = nn.Conv3d(1,1,3).to("cuda"); _ = _p(torch.zeros(1,1,4,4,4,device="cuda")).cpu(); DEVICE="cuda"; del _p
    except Exception as e:
        print("GPU present but conv3d unusable (P100/sm_60?) -> CPU:", str(e)[:80])
SCALE = np.array([1.625, 0.40625, 0.40625]); POOL = 4
WEIGHT_NAMES = ['unet3d_bright.pt', 'unet3d_traintophat.pt']
PREPROCS = ['', 'tophat']          # one per model; a model MUST use the preproc it was trained with
REPAIR = True
CAND_THR = 0.05
GAP_DT = 0
GAP_GATE_UM = 10.0
SNAP_UM = 3.0
SHORT_MIN = 4
LINEFIT_WEIGHT = 0.8
LINEFIT_WINDOW = 2
print("device:", DEVICE, "| torch", torch.__version__)
print("models:", list(zip(WEIGHT_NAMES, [p or "none" for p in PREPROCS])))
print("repair:", REPAIR, "| seed_thr:", 0.15, "| cand_thr:", CAND_THR, "| gap_dt:", GAP_DT, "| short:", SHORT_MIN)

CANDIROOT = ["/kaggle/input/biohub-cell-tracking-during-development",
             "/kaggle/input/competitions/biohub-cell-tracking-during-development", "data"]
ROOT = next((p for p in CANDIROOT if Path(p,"test").exists()), "data"); TEST_DIR = Path(ROOT)/"test"
def _find_weight(name):
    cands = [f"/kaggle/input/biohub-unet3d-weights/{name}", f"models/{name}"] +             glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    p = next((c for c in cands if Path(c).exists()), None)
    if p is None:
        raise FileNotFoundError(f"{name} not found; attach biohub-unet3d-weights. "
                                f"/kaggle/input: {glob.glob('/kaggle/input/*')}")
    return p
WEIGHTS = [_find_weight(n) for n in WEIGHT_NAMES]
OUT = "submission.csv"; print("data:", ROOT, "| weights:", WEIGHTS)

def _block(ci, co):
    return nn.Sequential(nn.Conv3d(ci,co,3,padding=1), nn.BatchNorm3d(co), nn.ReLU(inplace=True),
                         nn.Conv3d(co,co,3,padding=1), nn.BatchNorm3d(co), nn.ReLU(inplace=True))
class UNet3D(nn.Module):
    def __init__(self, base=24):
        super().__init__()
        self.e1=_block(1,base); self.e2=_block(base,base*2); self.e3=_block(base*2,base*4)
        self.pool=nn.MaxPool3d(2); self.bott=_block(base*4,base*8)
        self.u3=nn.ConvTranspose3d(base*8,base*4,2,stride=2); self.d3=_block(base*8,base*4)
        self.u2=nn.ConvTranspose3d(base*4,base*2,2,stride=2); self.d2=_block(base*4,base*2)
        self.u1=nn.ConvTranspose3d(base*2,base,2,stride=2); self.d1=_block(base*2,base)
        self.out=nn.Conv3d(base,1,1)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); b=self.bott(self.pool(e3))
        d3=self.d3(torch.cat([self.u3(b),e3],1)); d2=self.d2(torch.cat([self.u2(d3),e2],1))
        d1=self.d1(torch.cat([self.u1(d2),e1],1)); return self.out(d1)

MODELS = []
for _w in WEIGHTS:
    _ck = torch.load(_w, map_location=DEVICE)
    _m = UNet3D(base=_ck.get("base",24)).to(DEVICE); _m.load_state_dict(_ck["state_dict"]); _m.eval()
    MODELS.append(_m)
    print("loaded", Path(_w).name, "| val_recall", _ck.get("val_recall"), "| aug", _ck.get("aug"))
assert len(MODELS)==len(PREPROCS), (len(MODELS), len(PREPROCS))


## セル2: データ読み込み・前処理・ピーク検出のユーティリティ

**何をしているか**: Zarr形式の3D動画を1フレームずつ読み、XY方向を1/4に縮小（プーリング）してから正規化し、
U-Netの出力ヒートマップから細胞中心のピークを取り出す関数群を定義します。

**なぜそうするのか**:
- **`load_volume` が二重の実装になっている理由**: 通常は `zarr` ライブラリで読みますが、失敗した場合に備えて
  `blosc2` で圧縮チャンクを直接展開する経路が用意されています。Kaggleのオフライン環境ではライブラリのバージョン差で
  読めなくなることがあるため、**提出が0点になるリスクを潰す保険**です。
- **`pool_xy`（4x4平均プーリング）**: 生の解像度のまま3D U-Netを回すとメモリも時間も足りません。
  XYだけ縮めてZは縮めないのは、上で見た通り**Zがもともと粗いから**です。
- **`pool_norm` の正規化が percentile 50〜99.5 なのはなぜか**: 顕微鏡画像は撮影ごとに明るさが大きく変わります。
  最小値〜最大値で正規化すると、たった1個の異常に明るいノイズ点に引きずられて画像全体が暗くなります。
  **中央値を0、上位0.5%点を1**とすることで、外れ値に強い正規化になります。
- **`tophat`（トップハット変換）**: `grey_opening`（明るい小領域を消す処理）で作った「背景の推定」を引き算します。
  **ゆっくり変化する背景の明るさムラを消して、小さく明るい細胞だけを残す**古典的な手法です。
  このnotebookは「tophatをかけたモデル」と「かけないモデル」を両方使う点が重要で、
  *前処理そのものを多様性の源にしている*わけです。
- **`_refine`**: ピークの位置を周辺の重心で微調整します。プーリングで粗くした分の位置ずれを、
  サブボクセル精度で戻す狙いです。


In [ ]:

def read_array_meta(zp):
    with open(Path(zp)/"0"/"zarr.json") as f: m=json.load(f)
    return dict(shape=tuple(m["shape"]), dtype=np.dtype(m["data_type"]))
_ZC={}
def load_volume(zp, t, meta=None):
    try:
        import zarr; k=str(zp)
        if k not in _ZC: _ZC[k]=zarr.open(k,mode="r")["0"]
        return np.asarray(_ZC[k][t])
    except Exception:
        import blosc2
        if meta is None: meta=read_array_meta(zp)
        buf=blosc2.decompress(open(Path(zp)/"0"/"c"/str(t)/"0"/"0"/"0","rb").read())
        return np.frombuffer(buf,dtype=meta["dtype"]).reshape(meta["shape"][1:])

def pool_xy(vol, f=POOL):
    Z,Y,X=vol.shape; Y2,X2=(Y//f)*f,(X//f)*f
    v=vol[:,:Y2,:X2].astype(np.float32,copy=False)
    return v.reshape(Z,Y2//f,f,X2//f,f).mean(axis=(2,4))
def pool_norm(vol, preproc=""):
    p=pool_xy(vol)
    if preproc=="tophat":
        from scipy.ndimage import grey_opening
        p=np.clip(p-grey_opening(p,size=(1,7,7)),0.0,None)
    lo=float(np.percentile(p,50)); hi=float(np.percentile(p,99.5))
    return np.clip((p-lo)/(hi-lo+1e-6),-0.5,6.0).astype(np.float32)

def _refine(vol, zyx, rz=2, ryx=5):
    Z,Y,X=vol.shape; z,y,x=(int(round(v)) for v in zyx)
    z0,z1=max(0,z-rz),min(Z,z+rz+1); y0,y1=max(0,y-ryx),min(Y,y+ryx+1); x0,x1=max(0,x-ryx),min(X,x+ryx+1)
    crop=vol[z0:z1,y0:y1,x0:x1].astype(np.float32); bg=float(crop.min())
    w=np.clip(crop-bg,0,None); s=float(w.sum())
    if s<=0: return np.array([z,y,x],float),0.0
    zz,yy,xx=np.mgrid[z0:z1,y0:y1,x0:x1]
    return np.array([(zz*w).sum(),(yy*w).sum(),(xx*w).sum()])/s, float(crop.max()-bg)
def _physical_nms(coords, scores, radius_um, scale=SCALE):
    if len(coords)<=1: return coords,scores
    pts=coords*scale[None,:]; order=np.argsort(-scores); tree=cKDTree(pts)
    killed=np.zeros(len(coords),bool); keep=[]
    for i in order:
        if killed[i]: continue
        keep.append(int(i)); killed[tree.query_ball_point(pts[i],r=radius_um)]=True
    keep=np.array(keep); return coords[keep],scores[keep]

UNET_THRESH=0.15; NMS_UM=4.0
DETECT_THRESH = min(UNET_THRESH, CAND_THR) if REPAIR else UNET_THRESH
def detect(vol):
    # each model sees the preprocessing it was TRAINED with, then AVERAGE the heatmaps
    # (never union the detections: DoG-union U-Net measured 0.661 -- over-detection
    #  blows past T_true and the node-count adjustment punishes it)
    hs=[]
    for _m,_pp in zip(MODELS, PREPROCS):
        x=pool_norm(vol,_pp)
        with torch.no_grad():
            hs.append(torch.sigmoid(_m(torch.from_numpy(x)[None,None].to(DEVICE)))[0,0].float().cpu().numpy())
    h = hs[0] if len(hs)==1 else np.mean(hs, axis=0)
    pk=peak_local_max(h, min_distance=1, threshold_abs=DETECT_THRESH, exclude_border=False)
    if len(pk)==0: return np.zeros((0,3)), np.zeros(0)
    sc=h[pk[:,0],pk[:,1],pk[:,2]].astype(float)
    coords=pk.astype(float); coords[:,1]=coords[:,1]*POOL+(POOL-1)/2; coords[:,2]=coords[:,2]*POOL+(POOL-1)/2
    ref=np.array([_refine(vol,c)[0] for c in coords])
    return _physical_nms(ref, sc, NMS_UM)


## セル3: リンク（対応付け）とグラフ後処理の中核

**何をしているか**: フレーム t の細胞集合と t+1 の細胞集合を**1対1で対応付ける**関数 `_link` と、
ギャップ補完・短小トラック除去などのグラフ処理を定義します。

**なぜそうするのか**:
- **ハンガリー法（`linear_sum_assignment`）**: 「どの細胞をどの細胞につなぐか」は**割当問題**です。
  貪欲に近い順から結ぶと、1個の判断ミスが連鎖して全体が崩れます。ハンガリー法は
  **全体のコスト合計が最小になる組み合わせを一発で求める**ので、局所的な誘惑に負けません。
- **2段階ゲート（`TIGHT_UM=6.0` → `MAX_LINK_UM=10.0`）**: まず「6µm以内」という厳しい条件だけで確実なペアを確定し、
  余った細胞だけを「10µm以内」で救済します。**自信のあるペアを先に固定してしまう**ことで、
  遠くの怪しいペアが確実なペアを横取りするのを防ぎます。
- **速度による予測位置（`pred = P + 0.5*prev_vel`）**: 直前の動きから次の位置を予測し、**予測位置との距離**をコストにします。
  ただし**ゲート判定は生の距離 `Draw` で行う**という細かい工夫があります。予測が外れた時に
  「予測位置には近いが実際には遠い」ペアを許してしまわないための安全弁です。
- **`_gap_support`（ギャップ補完）**: 細胞が1〜数フレーム検出できなかったとき、その間を線形補間した位置に
  **低い閾値で拾った弱い候補があるか**を確認し、あればトラックをつなぎ直します。
  「見えなかった」と「本当にいなかった」を区別する仕掛けです。
- **`_short_filter`**: 数フレームしか続かないトラックは**ほぼノイズ**なので捨てます。Jaccard系の指標では
  余計なノードもエッジも減点対象なので、確信のないものは出さないほうが得です。


In [ ]:

MAX_LINK_UM=10.0; TIGHT_UM=6.0
def _link(prev_xyz, curr_xyz, prev_vel):
    if len(prev_xyz)==0 or len(curr_xyz)==0: return []
    P=prev_xyz*SCALE[None,:]; C=curr_xyz*SCALE[None,:]
    pred=P+(0.5*prev_vel if prev_vel is not None else 0.0); N,M=len(P),len(C); BIG=1e9
    def _hun(pi,ci,gate):
        if len(pi)==0 or len(ci)==0: return []
        Draw=np.sqrt(((P[pi][:,None]-C[ci][None])**2).sum(2)); D=np.sqrt(((pred[pi][:,None]-C[ci][None])**2).sum(2))
        cost=np.where(Draw>gate,BIG,D); ri,rc=linear_sum_assignment(cost)
        return [(int(pi[r]),int(ci[c])) for r,c in zip(ri,rc) if cost[r,c]<BIG]
    links=_hun(np.arange(N),np.arange(M),min(TIGHT_UM,MAX_LINK_UM))
    up={p for p,_ in links}; uc={c for _,c in links}
    fp=np.array([i for i in range(N) if i not in up],int); fc=np.array([j for j in range(M) if j not in uc],int)
    return links+_hun(fp,fc,MAX_LINK_UM)

COLS=["dataset","row_type","node_id","t","z","y","x","source_id","target_id"]
def _dist_um(a,b):
    d=(np.asarray(a,float)-np.asarray(b,float))*SCALE
    return float(np.sqrt((d*d).sum()))

def _gap_support(pe, ps, te, dt, cand, cand_trees):
    out=[]
    for k in range(1, dt):
        tk=te+k; interp=pe+(ps-pe)*(k/dt); tree=cand_trees[tk] if 0 <= tk < len(cand_trees) else None
        if tree is None: return None
        dist,idx=tree.query(interp*SCALE)
        if dist > SNAP_UM: return None
        out.append(cand[tk][idx])
    return out

def _segments(nodes, succ, pred, vel):
    segs=[]
    for g in nodes:
        if g in pred: continue
        ch=[g]
        while ch[-1] in succ: ch.append(succ[ch[-1]])
        segs.append(ch)
    ends=[]; starts=[]
    for ch in segs:
        ge=ch[-1]; ve=vel.get(ge, np.zeros(3))
        if len(ch) >= 2: ve=(nodes[ch[-1]]["xyz"]-nodes[ch[-2]]["xyz"])*SCALE
        ends.append((ge, int(nodes[ge]["t"]), nodes[ge]["xyz"], ve))
        gs=ch[0]; starts.append((gs, int(nodes[gs]["t"]), nodes[gs]["xyz"]))
    return segs, ends, starts

def _gap_close(nodes, edges, succ, pred, vel, cand, cand_trees, next_id):
    if GAP_DT <= 0: return next_id
    segs, ends, starts = _segments(nodes, succ, pred, vel)
    seglen=[len(ch) for ch in segs]; props=[]
    for i,(ge,te,pe,ve_um) in enumerate(ends):
        for j,(gs,ts,ps) in enumerate(starts):
            dt=ts-te
            if i == j or dt < 1 or dt > GAP_DT: continue
            predpos=pe+(ve_um/SCALE)*dt
            cost=_dist_um(predpos, ps)
            if cost > GAP_GATE_UM: continue
            if dt >= 2 and _gap_support(pe, ps, te, dt, cand, cand_trees) is None: continue
            props.append((cost,i,j,dt))
    used_e=set(); used_s=set()
    for _,i,j,dt in sorted(props):
        if i in used_e or j in used_s: continue
        used_e.add(i); used_s.add(j)
        ge,te,pe,_=ends[i]; gs,_,ps=starts[j]
        if dt == 1:
            edges.append((ge,gs)); continue
        prev=ge
        for k in range(1,dt):
            tk=te+k; interp=pe+(ps-pe)*(k/dt); use=interp
            if cand_trees[tk] is not None:
                dist,idx=cand_trees[tk].query(interp*SCALE)
                if dist <= SNAP_UM: use=cand[tk][idx]
            ng=next_id; next_id += 1
            nodes[ng]={"t":tk, "xyz":np.asarray(use,float)}
            edges.append((prev,ng)); prev=ng
        edges.append((prev,gs))
    return next_id

def _short_filter(nodes, edges):
    if SHORT_MIN <= 1 or not edges: return nodes, edges
    parent={nid:nid for nid in nodes}
    def find(x):
        while parent[x] != x:
            parent[x]=parent[parent[x]]; x=parent[x]
        return x
    def union(a,b):
        if a not in parent or b not in parent: return
        ra,rb=find(a),find(b)
        if ra != rb: parent[ra]=rb
    out_count=defaultdict(int)
    for a,b in edges:
        union(a,b); out_count[a]+=1
    comps=defaultdict(list)
    for nid in nodes: comps[find(nid)].append(nid)
    keep=set()
    for members in comps.values():
        has_div=any(out_count[n] >= 2 for n in members)
        if len(members) >= SHORT_MIN or has_div: keep.update(members)
    nodes2={nid:n for nid,n in nodes.items() if nid in keep}
    edges2=[(a,b) for a,b in edges if a in nodes2 and b in nodes2]
    return nodes2, edges2

def _linefit(nodes, edges):
    if LINEFIT_WEIGHT <= 0: return
    pred=defaultdict(list); succ=defaultdict(list)
    for a,b in edges:
        if a in nodes and b in nodes and int(nodes[b]["t"]) == int(nodes[a]["t"]) + 1:
            succ[a].append(b); pred[b].append(a)
    orig={k:v["xyz"].copy() for k,v in nodes.items()}; updates={}
    W=int(LINEFIT_WINDOW)
    for nid in nodes:
        neigh=[(0,nid)]; cur=nid
        for step in range(1,W+1):
            ps=pred.get(cur, [])
            if len(ps) != 1: break
            cur=ps[0]; neigh.append((-step,cur))
        cur=nid
        for step in range(1,W+1):
            ss=succ.get(cur, [])
            if len(ss) != 1: break
            cur=ss[0]; neigh.append((step,cur))
        if len(neigh) < 3: continue
        dt=np.array([a for a,_ in neigh], float)
        xyz=np.stack([orig[n] for _,n in neigh])
        fit=np.array([np.polyval(np.polyfit(dt, xyz[:,ax], 1), 0.0) for ax in range(3)])
        if np.isfinite(fit).all():
            updates[nid]=(1.0-LINEFIT_WEIGHT)*orig[nid]+LINEFIT_WEIGHT*fit
    for nid,xyz in updates.items(): nodes[nid]["xyz"]=xyz

def _emit(ds, nodes, edges):
    edge_set=[]; seen=set()
    for a,b in edges:
        if a == b or a not in nodes or b not in nodes or (a,b) in seen: continue
        seen.add((a,b)); edge_set.append((a,b))
    used=set()
    for a,b in edge_set: used.add(a); used.add(b)
    nrows=[]; erows=[]
    for nid in sorted(used):
        n=nodes[nid]; z,y,x=n["xyz"]
        nrows.append((ds,"node",int(nid),int(n["t"]),float(z),float(y),float(x),-1,-1))
    for a,b in edge_set:
        if a in used and b in used: erows.append((ds,"edge",-1,-1,-1,-1,-1,int(a),int(b)))
    return pd.DataFrame(nrows,columns=COLS), pd.DataFrame(erows,columns=COLS)

def repair_track(dets, ds):
    nodes={}; frame_ids=[]; cand=[]; cand_trees=[]; nid=1
    for t,(coords,scores) in enumerate(dets):
        coords=np.asarray(coords,float).reshape(-1,3); scores=np.asarray(scores,float).reshape(-1)
        seeds=coords[scores >= UNET_THRESH]
        cands=coords[(scores >= CAND_THR) & (scores < UNET_THRESH)]
        cand.append(cands); cand_trees.append(cKDTree(cands*SCALE) if len(cands) else None)
        ids=[]
        for xyz in seeds:
            nodes[nid]={"t":t, "xyz":np.asarray(xyz,float)}; ids.append(nid); nid += 1
        frame_ids.append(ids)
    edges=[]; succ={}; pred={}; vel={}
    for t in range(len(dets)-1):
        P=np.asarray([nodes[g]["xyz"] for g in frame_ids[t]], float).reshape(-1,3)
        C=np.asarray([nodes[g]["xyz"] for g in frame_ids[t+1]], float).reshape(-1,3)
        if len(P) == 0 or len(C) == 0: continue
        prev_vel=np.array([vel.get(g, np.zeros(3)) for g in frame_ids[t]])
        for pi,ci in _link(P, C, prev_vel if len(prev_vel) else None):
            gp,gc=frame_ids[t][pi],frame_ids[t+1][ci]
            edges.append((gp,gc)); succ[gp]=gc; pred[gc]=gp; vel[gc]=(C[ci]-P[pi])*SCALE
    nid=_gap_close(nodes, edges, succ, pred, vel, cand, cand_trees, nid)
    nodes,edges=_short_filter(nodes, edges)
    _linefit(nodes, edges)
    return _emit(ds, nodes, edges)

def track_movie(zp, ds, T):
    if REPAIR:
        meta=read_array_meta(zp); dets=[]
        for t in range(T):
            dets.append(detect(load_volume(zp,t,meta))); gc.collect()
        return repair_track(dets, ds)
    meta=read_array_meta(zp); node_rows=[]; edge_rows=[]
    prev_ids=[]; prev_xyz=np.zeros((0,3)); prev_vel=None; nid=1
    for t in range(T):
        coords,scores=detect(load_volume(zp,t,meta)); gc.collect()
        ids=list(range(nid,nid+len(coords))); nid+=len(coords)
        for i,c in zip(ids,coords): node_rows.append((ds,"node",i,t,float(c[0]),float(c[1]),float(c[2]),-1,-1))
        if t>0 and len(prev_ids):
            links=_link(prev_xyz,coords,prev_vel); vel=np.zeros((len(prev_xyz),3))
            for p,c in links:
                edge_rows.append((ds,"edge",-1,-1,-1,-1,-1,prev_ids[p],ids[c])); vel[p]=(coords[c]-prev_xyz[p])*SCALE
            nv=np.zeros((len(coords),3))
            for p,c in links: nv[c]=vel[p]
            prev_vel=nv
        else: prev_vel=None
        prev_ids,prev_xyz=ids,coords
    nodes=pd.DataFrame(node_rows,columns=COLS); edges=pd.DataFrame(edge_rows,columns=COLS)
    if len(edges):
        used=set(edges.source_id)|set(edges.target_id); nodes=nodes[nodes.node_id.isin(used)].reset_index(drop=True)
    return nodes,edges

def avail_T(zp):
    meta=read_array_meta(zp); T=meta["shape"][0]
    present=[t for t in range(T) if (Path(zp)/"0"/"c"/str(t)/"0"/"0"/"0").exists()]
    return max(present)+1 if present else 0



## セル4: 3モデル×4方向フリップTTA — 「平均するのは確率ではなくロジット」

**何をしているか**: 3つの3D U-Net（bright / tophat / v2-tophat-b32）それぞれについて、
入力を4通り（そのまま・Y反転・X反転・両方反転）に変えて推論し、**逆変換してから平均**します。
最後に3つのブランチのヒートマップを等重みで平均します。

**なぜそうするのか**:
- **TTA（Test-Time Augmentation）とは**: 学習時に使ったのと同じ種類の変形を推論時にも適用し、結果を平均する手法。
  1回の推論のたまたまの当たり外れをならして、**取りこぼし（FN）を減らす**のが目的です。
- **Zを反転しないのはなぜか**: Z方向は解像度が違う（異方性）ため、学習時にもZ反転は使っていません。
  **学習時に見たことのない変形をTTAで入れると、かえって精度が落ちます**。
  「TTAは学習時の拡張と揃える」は実務でも重要な原則です。
- **★ 最重要: `sigmoid` の前にロジットを平均している**。
  sigmoidは非線形なので、`mean(sigmoid(x))` と `sigmoid(mean(x))` は違う値になります。
  確率にしてから平均すると、0や1に張り付いた自信満々の予測が平均を支配してしまいます。
  **ロジット（sigmoid前の生の値）で平均**すれば各ビューが対等に効き、確率空間での平均より素直な合成になります。
  この「ロジット空間で混ぜる」発想はテーブルコンペのスタッキングでもまったく同じ理由で使われます。
- **前処理の違いを多様性にする**: 3ブランチのうち2つは `tophat` あり、1つはなし。
  同じモデルのシード違いを増やすより、**間違え方が違うモデルを混ぜる**ほうがアンサンブルは効きます。
  ここでは重みファイルではなく**前処理**で違いを作っています。
- `SHORT_MIN = 4 → 6`: TTAとアンサンブルで検出が増えた分、ノイズ由来の短いトラックも増えます。
  そのため足切りを厳しくしています。**片方だけ変えるのではなく、影響が及ぶ側も一緒に調整する**という良い例です。


In [ ]:
# ======================================================================
# DETECTION VARIANT — N-model ensemble + flip-quartet TTA on pooled (Y,X)
# ----------------------------------------------------------------------
# Per branch: forward the 4 flip views {id, flipY, flipX, flipY.flipX} — the
# exact augmentations used in training (Z never touched: anisotropic) —
# inverse-transform the LOGITS (flips are self-inverse), average, sigmoid
# AFTER averaging. Branch heatmaps averaged (equal weights). Everything
# downstream (peak_local_max @0.15, refine, physical NMS 4.0 um) unchanged.
# ======================================================================
_BRANCHES = []
_BRANCHES.append(([MODELS[0]], [1.0], ''))
_BRANCHES.append(([MODELS[1]], [1.0], 'tophat'))
_ck = torch.load(_find_weight('unet3d_v2_tophat_b32.pt'), map_location=DEVICE)
_m = UNet3D(base=_ck.get('base', 24)).to(DEVICE)
_m.load_state_dict(_ck['state_dict']); _m.eval()
print('loaded', 'unet3d_v2_tophat_b32.pt', '| val_recall', _ck.get('val_recall'))
_BRANCHES.append(([_m], [1.0], 'tophat'))
# SHORT_MIN override: read by _short_filter (cell 3) at call time.
SHORT_MIN = 6
_FLIP_VIEWS = (0, 1, 2, 3)  # bit0 = flip Y, bit1 = flip X

def _flip(x, i):
    if i & 1: x = np.flip(x, -2)
    if i & 2: x = np.flip(x, -1)
    return x

detect_base = detect  # base detect kept for reference (A/B)
def detect(vol):
    hs = []
    for models, ws, pp in _BRANCHES:
        x = pool_norm(vol, pp)
        acc = None
        for i in _FLIP_VIEWS:
            xv = np.ascontiguousarray(_flip(x, i))
            with torch.no_grad():
                lg = None
                for m, w in zip(models, ws):
                    l = m(torch.from_numpy(xv)[None, None].to(DEVICE))[0, 0].float().cpu().numpy()
                    lg = w * l if lg is None else lg + w * l
            lg = _flip(lg, i)  # self-inverse
            acc = lg if acc is None else acc + lg
        hs.append(1.0 / (1.0 + np.exp(-(acc / len(_FLIP_VIEWS)))))
    h = hs[0] if len(hs) == 1 else np.mean(hs, axis=0)
    pk = peak_local_max(h, min_distance=1, threshold_abs=DETECT_THRESH, exclude_border=False)
    if len(pk) == 0: return np.zeros((0, 3)), np.zeros(0)
    sc = h[pk[:, 0], pk[:, 1], pk[:, 2]].astype(float)
    coords = pk.astype(float)
    coords[:, 1] = coords[:, 1] * POOL + (POOL - 1) / 2
    coords[:, 2] = coords[:, 2] * POOL + (POOL - 1) / 2
    ref = np.array([_refine(vol, c)[0] for c in coords])
    return _physical_nms(ref, sc, NMS_UM)


## セル5: 後処理パッチ — 「1対1の限界」を後から埋める

**何をしているか**: リンク済みのグラフに対して2つの保守的なパッチを当てます。
**PATCH 1 = 安全な分裂の復元**、**PATCH 2 = 1フレームだけのギャップのスナップ**です。検出結果は一切変更しません。

**なぜそうするのか**:
- **ハンガリー法の構造的な弱点**: 線形割当は必ず**1対1**なので、親1個から娘2個が生まれる分裂は
  **原理的に片方しかリンクできません**。作者の計測では、パッチ前の分裂検出は TP/FP/FN = 0/0/14、
  つまり**分裂を1個も検出できていません**。モデルが悪いのではなく、リンクの数学的な形が分裂を許していないのです。
  「スコアが伸びない原因が、モデルではなくアルゴリズムの構造にある」ことを見抜いた良い例です。
- **どう救済するか（条件の読み方）**: 親 p が t で子 c1 だけを持つとき、t+1 の**孤児ノード（親がいないノード）** q を
  2人目の娘の候補にします。採用するのは以下を全部満たすときだけ:
  - `d(p,q) ≤ 12µm`, `d(c1,q) ≤ 15µm`, `d(p,c1) ≤ 10µm` — 距離が生物学的にあり得る範囲か
  - **q と c1 が互いに最近傍**であること — 姉妹細胞は互いに一番近いはず、という制約
  - **分裂後に離れていく**こと: `d(succ(c1), succ(q)) - d(c1,q) ≥ 2.25µm` — 次のフレームで距離が広がっているか。
    これが一番賢い条件です。**たまたま近くにいる無関係な2細胞**と、**今まさに分裂して離れていく姉妹**を、
    「その後どうなったか」という未来の情報で区別しています。
- **なぜここまで条件を厳しくするのか**: 分裂を1つ当てるより、**間違った分裂を1つ作るほうが指標へのダメージが大きい**からです。
  実際 FP は 0→31 に増えましたが、それでも FN が 14→8 に減り、正味プラス（0.8387 → 0.8516）でした。
  **「増やす副作用込みで検証してから採用する」**という手順そのものが、このセルの一番の学びです。
- **PATCH 2 は「スナップのみ」**: 1フレームのギャップを、新しいノードを作らずに既存ノードへ寄せるだけ。
  効果は +0.0002 と小さいですが、**副作用がほぼゼロ**なので採用されています。


In [ ]:
# ======================================================================
# POST-LINK GRAPH PATCHES  (final config "final_combo", VAL-24 verified)
# ----------------------------------------------------------------------
# Two conservative patches on the linker GRAPH (nodes + edges) only.
# Detection is UNTOUCHED: 2-model heatmap-mean ensemble @ UNET_THRESH=0.15,
# each model with its own preprocessing, physical NMS 4.0 um.
#
# VAL-24, official tracking_cellmot metric (24 held-out movies):
#   base 0.8387  ->  +safe divisions 0.8516  ->  +gap(snap-only) 0.8518
#   division TP/FP/FN = 6/31/8   (base M001: 0/0/14 — predicts no divisions)
#
# PATCH 1 — SAFE DIVISIONS (after short-filter + linefit, before emit).
#   The base linker is strictly 1:1, so a mitosis second daughter is never
#   linked. For a parent p (frame t) with exactly ONE child c1 at t+1, propose
#   a second daughter q among ORPHAN (in-degree 0) nodes at t+1:
#     d(p,q) <= 12 um,  d(c1,q) <= 15 um,  d(p,c1) <= 10 um (sanity),
#     q is the nearest orphan to c1 as well (sisters are mutual nearest
#     orphans), and the sisters DIVERGE after mitosis:
#     d(succ(c1),succ(q)) - d(c1,q) >= 2.25 um (both children continue at t+2).
#   Accepted 1:1 greedy by score = d(p,q) + 0.15*d(c1,q); safety caps
#   (0.76% of frame nodes, 0.375% of movie nodes) — never hit in practice.
#
# PATCH 2 — 1-FRAME GAP CLOSING, snap-only (before short-filter).
#   Track END at t vs track START at t+2, Hungarian under a 9 um gate; ends
#   must have a predecessor and starts a successor (no fragment tips). The
#   t+1 midpoint is filled ONLY by snapping to an unused low-score candidate
#   (0.10 <= score < UNET_THRESH) within 3.2 um — never a synthetic point.
#   Cap 0.3% of movie nodes.
# ======================================================================
DIV_PARENT_UM=12.0; DIV_SISTER_UM=15.0; DIV_CHILD_UM=10.0; DIV_DIVERGE_UM=2.25
DIV_W_SISTER=0.15; DIV_FRAME_CAP=0.0076; DIV_GLOBAL_CAP=0.00375
GAP1_GATE_UM=9.0; GAP1_SNAP_UM=3.2; GAP1_MIN_CAND_SCORE=0.10; GAP1_CAP_FRAC=0.003

def _add_safe_divisions(nodes, edges):
    succ=defaultdict(list); pred=defaultdict(list)
    for a,b in edges:
        succ[a].append(b); pred[b].append(a)
    by_t=defaultdict(list)
    for nid,n in nodes.items(): by_t[n["t"]].append(nid)
    orphan_tree={}; orphan_ids={}
    for t,ids in by_t.items():
        orph=[g for g in ids if not pred.get(g)]
        orphan_ids[t]=orph
        if orph: orphan_tree[t]=cKDTree(np.asarray([nodes[g]["xyz"] for g in orph],float)*SCALE)
    proposals=[]
    for p,n in nodes.items():
        ch=succ.get(p,[])
        if len(ch)!=1: continue
        c1=ch[0]; t=n["t"]
        if nodes[c1]["t"]!=t+1: continue
        if len(pred.get(p,[]))!=1: continue                       # parent must have a predecessor
        if _dist_um(n["xyz"],nodes[c1]["xyz"])>DIV_CHILD_UM: continue
        tree=orphan_tree.get(t+1)
        if tree is None: continue
        for idx in tree.query_ball_point(n["xyz"]*SCALE, r=DIV_PARENT_UM):
            q=orphan_ids[t+1][idx]
            if q==c1: continue
            d_pq=_dist_um(n["xyz"],nodes[q]["xyz"]); d_s=_dist_um(nodes[c1]["xyz"],nodes[q]["xyz"])
            if d_s>DIV_SISTER_UM: continue
            d_q,q_idx=tree.query(nodes[c1]["xyz"]*SCALE)          # sisters: mutual nearest orphans
            if orphan_ids[t+1][q_idx]!=q or d_q>DIV_SISTER_UM: continue
            c1n,qn=succ.get(c1,[]),succ.get(q,[])                 # both children continue at t+2
            if len(c1n)!=1 or len(qn)!=1: continue
            if _dist_um(nodes[c1n[0]]["xyz"],nodes[qn[0]]["xyz"])-d_s<DIV_DIVERGE_UM: continue
            proposals.append((d_pq+DIV_W_SISTER*d_s,t,p,q))
    proposals.sort()
    used_p=set(); used_q=set(); per_frame=defaultdict(int)
    n_frame={t:len(ids) for t,ids in by_t.items()}
    max_global=max(1,int(DIV_GLOBAL_CAP*len(nodes))); added=0
    for score,t,p,q in proposals:
        if added>=max_global: break
        if p in used_p or q in used_q: continue
        if per_frame[t]>=max(1,int(DIV_FRAME_CAP*n_frame[t])): continue
        edges.append((p,q)); succ[p].append(q); pred[q].append(p)
        used_p.add(p); used_q.add(q); per_frame[t]+=1; added+=1
    return added

def _gap_close_1f_snap(nodes, edges, cand, cand_sc, cand_trees, next_id):
    T=len(cand); succ={}; pred={}
    for a,b in edges:
        succ[a]=b; pred[b]=a
    by_t=defaultdict(list)
    for nid,n in nodes.items(): by_t[n["t"]].append(nid)
    cand_used=[np.zeros(len(c),bool) for c in cand]
    max_gaps=max(1,int(GAP1_CAP_FRAC*len(nodes))); n_added=0; BIG=1e9
    for t in range(T-2):
        if n_added>=max_gaps: break
        ends=[g for g in by_t.get(t,[]) if g not in succ and g in pred]      # track len >= 2 back
        starts=[g for g in by_t.get(t+2,[]) if g not in pred and g in succ]  # track len >= 2 forward
        if not ends or not starts: continue
        P=np.asarray([nodes[g]["xyz"] for g in ends],float)*SCALE
        C=np.asarray([nodes[g]["xyz"] for g in starts],float)*SCALE
        D=np.sqrt(((P[:,None]-C[None])**2).sum(2)); cost=np.where(D>GAP1_GATE_UM,BIG,D)
        ri,ci=linear_sum_assignment(cost)
        props=sorted((float(D[r,c]),ends[r],starts[c]) for r,c in zip(ri,ci) if cost[r,c]<BIG)
        for d,ge,gs in props:
            if n_added>=max_gaps: break
            if ge in succ or gs in pred: continue
            mid=0.5*(nodes[ge]["xyz"]+nodes[gs]["xyz"]); use=None
            tree=cand_trees[t+1]
            if tree is not None:
                dist,idx=tree.query(mid*SCALE)
                if (dist<=GAP1_SNAP_UM and not cand_used[t+1][idx]
                        and cand_sc[t+1][idx]>=GAP1_MIN_CAND_SCORE):
                    use=np.asarray(cand[t+1][idx],float); cand_used[t+1][idx]=True
            if use is None: continue                                       # snap-only: never synthesise
            ng=next_id; next_id+=1
            nodes[ng]={"t":t+1,"xyz":use}
            edges.append((ge,ng)); edges.append((ng,gs))
            succ[ge]=ng; pred[ng]=ge; succ[ng]=gs; pred[gs]=ng
            n_added+=1
    return next_id, n_added


# appearance cost in the linker: cost += 2.0 * |logit(s_prev) - logit(s_curr)| (um)
LINK_SIM_W = 2.0
def _logit(s):
    s = np.clip(s, 1e-4, 1 - 1e-4); return np.log(s / (1 - s))

def _link_sim(prev_xyz, curr_xyz, prev_vel, prev_sc, curr_sc):
    if len(prev_xyz)==0 or len(curr_xyz)==0: return []
    P=prev_xyz*SCALE[None,:]; C=curr_xyz*SCALE[None,:]
    pred=P+(0.5*prev_vel if prev_vel is not None else 0.0); N,M=len(P),len(C); BIG=1e9
    sim = LINK_SIM_W*np.abs(_logit(prev_sc)[:,None]-_logit(curr_sc)[None])
    def _hun(pi,ci,gate):
        if len(pi)==0 or len(ci)==0: return []
        Draw=np.sqrt(((P[pi][:,None]-C[ci][None])**2).sum(2)); D=np.sqrt(((pred[pi][:,None]-C[ci][None])**2).sum(2))
        cost=np.where(Draw>gate,BIG,D)+np.where(Draw>gate,0.0,sim[np.ix_(pi,ci)])
        ri,rc=linear_sum_assignment(cost)
        return [(int(pi[r]),int(ci[c])) for r,c in zip(ri,rc) if cost[r,c]<BIG]
    links=_hun(np.arange(N),np.arange(M),min(TIGHT_UM,MAX_LINK_UM))
    up={p for p,_ in links}; uc={c for _,c in links}
    fp=np.array([i for i in range(N) if i not in up],int); fc=np.array([j for j in range(M) if j not in uc],int)
    return links+_hun(fp,fc,MAX_LINK_UM)

repair_track_base=repair_track   # base pipeline kept for reference (A/B)
def repair_track(dets, ds):
    # same graph pipeline as the base kernel, with the two patch hooks marked
    nodes={}; frame_ids=[]; cand=[]; cand_sc=[]; cand_trees=[]; nid=1; NSC={}
    for t,(coords,scores) in enumerate(dets):
        coords=np.asarray(coords,float).reshape(-1,3); scores=np.asarray(scores,float).reshape(-1)
        seeds=coords[scores >= UNET_THRESH]
        cm=(scores >= CAND_THR) & (scores < UNET_THRESH)
        cand.append(coords[cm]); cand_sc.append(scores[cm])                # cand scores needed by patch 2
        cand_trees.append(cKDTree(cand[-1]*SCALE) if len(cand[-1]) else None)
        seed_sc=scores[scores >= UNET_THRESH]
        ids=[]
        for xyz,s_ in zip(seeds,seed_sc):
            nodes[nid]={"t":t, "xyz":np.asarray(xyz,float)}; NSC[nid]=float(s_); ids.append(nid); nid += 1
        frame_ids.append(ids)
    edges=[]; succ={}; pred={}; vel={}
    for t in range(len(dets)-1):
        P=np.asarray([nodes[g]["xyz"] for g in frame_ids[t]], float).reshape(-1,3)
        C=np.asarray([nodes[g]["xyz"] for g in frame_ids[t+1]], float).reshape(-1,3)
        if len(P)==0 or len(C)==0: continue
        prev_vel=np.array([vel.get(g, np.zeros(3)) for g in frame_ids[t]])
        psc=np.array([NSC[g] for g in frame_ids[t]]); csc=np.array([NSC[g] for g in frame_ids[t+1]])
        for pi,ci in _link_sim(P, C, prev_vel if len(prev_vel) else None, psc, csc):
            gp,gc=frame_ids[t][pi],frame_ids[t+1][ci]
            edges.append((gp,gc)); succ[gp]=gc; pred[gc]=gp; vel[gc]=(C[ci]-P[pi])*SCALE
    nid,n_gaps=_gap_close_1f_snap(nodes, edges, cand, cand_sc, cand_trees, nid)   # PATCH 2 (pre short-filter)
    nodes,edges=_short_filter(nodes, edges)
    _linefit(nodes, edges)
    n_div=_add_safe_divisions(nodes, edges)                                       # PATCH 1 (post linefit)
    print(f"    patches[{ds}]: gaps={n_gaps} divs={n_div}")
    return _emit(ds, nodes, edges)


## セル6: 全動画を処理して submission.csv を書き出す

**何をしているか**: テスト用の各 `.zarr` 動画に対して `track_movie` を実行し、ノード行とエッジ行を結合して1本のCSVにします。

**なぜそうするのか**:
- **1動画ずつ処理してすぐ結合する**理由は**メモリ**です。3D動画を全部同時に開くとRAMが足りません。
- **`assert list(sub.columns) == exp`** に注目してください。提出直前に列の順番と名前を検証しています。
  Kaggleのコードコンペでは**列名が1つ違うだけで提出がエラー**になり、長時間の推論が無駄になります。
  こうした**フェイルクローズドな検証**（間違っていたら止まる）は、地味ですが最も費用対効果の高い作法です。
- 各動画のノード数・エッジ数・処理時間を毎回printしているのも重要で、
  異常に多い／少ない動画があればログから即座に気づけます。


In [ ]:
parts=[]
for zp in sorted(TEST_DIR.glob("*.zarr")):
    ds=zp.name.replace(".zarr",""); T=avail_T(zp)
    if T==0: print("skip",ds); continue
    t0=time.time(); nodes,edges=track_movie(zp,ds,T); parts+=[nodes,edges]
    print(f"  {ds}: T={T} nodes={len(nodes)} edges={len(edges)} ({time.time()-t0:.1f}s)")
sub=pd.concat(parts,ignore_index=True); sub.index.name="id"; sub.to_csv(OUT)
exp=["dataset","row_type","node_id","t","z","y","x","source_id","target_id"]; assert list(sub.columns)==exp
print("wrote",OUT,"rows",len(sub),"| nodes",(sub.row_type=='node').sum(),"edges",(sub.row_type=='edge').sum())
